![Banner](banner.jpg)

# Laboratorio 10: Redes Recurrentes — LSTM y GRU

## 1. Introducción

### ¿Qué son las Redes Neuronales Recurrentes (RNN)?

Las **redes neuronales recurrentes** son una familia de arquitecturas de deep learning diseñadas
para procesar **datos secuenciales**, como series de tiempo, texto o audio. A diferencia de las
redes feedforward tradicionales, las RNN mantienen un **estado oculto** que actúa como una
"memoria" de los pasos anteriores en la secuencia.

Sin embargo, las RNN simples sufren del problema del **desvanecimiento del gradiente**
(vanishing gradient), lo que dificulta aprender dependencias a largo plazo. Para resolver
esto, se han propuesto dos arquitecturas clave:

### LSTM (Long Short-Term Memory)

Las redes **LSTM** fueron propuestas por Hochreiter & Schmidhuber (1997). Su idea central
es agregar un **estado de celda** (cell state) que fluye a lo largo de la secuencia,
regulado por tres compuertas:

1. **Forget gate**: decide qué información descartar del estado de celda.
2. **Input gate**: decide qué información nueva agregar al estado de celda.
3. **Output gate**: decide qué parte del estado de celda se usa como salida.

Gracias a estas compuertas, las LSTM pueden aprender **dependencias a largo plazo**
sin sufrir tanto del desvanecimiento del gradiente.

### GRU (Gated Recurrent Unit)

Las redes **GRU** (Cho et al., 2014) son una variante simplificada de las LSTM. En lugar
de tres compuertas, usan solo dos:

1. **Reset gate**: controla cuánto del estado anterior se olvida.
2. **Update gate**: controla cuánto del nuevo estado se mezcla con el anterior.

Las GRU tienen **menos parámetros** que las LSTM, lo que las hace más rápidas de entrenar
y, en muchos casos, logran un desempeño similar.

### Objetivo de esta práctica

En este laboratorio usaremos datos financieros de acciones tecnológicas para:
- Construir y entrenar un modelo **LSTM** para predicción de series de tiempo.
- Evaluar su desempeño en la predicción del precio de cierre de una acción.
- Visualizar las predicciones vs. valores reales.

## 2. Preparación del entorno

In [ ]:
!pip install yfinance pandas-datareader

In [ ]:
# para settear tiempos
from datetime import datetime

import pandas as pd
import yfinance as yf

## 3. Carga del dataset

Para este problema usaremos la librería **yfinance**, la cual retorna los valores
de stock para diferentes empresas. En este caso tenemos los valores de:
- **AAPL** (Apple)
- **GOOG** (Google/Alphabet)
- **MSFT** (Microsoft)
- **AMZN** (Amazon)

Usaremos los registros de los últimos **3 años** como ejemplo.

In [ ]:
tech_list = ['AAPL', 'GOOG', 'MSFT', 'AMZN']

# Asignamos el rango de fechas en las que queremos consultar los datos
end = datetime(year=2021, month=12, day=31)
start = datetime(end.year - 3, end.month, end.day)

tech_stock = yf.download(tech_list, start, end)

In [ ]:
tech_stock = pd.DataFrame(
    {f'{price_type}_{ticker}': tech_stock[(price_type, ticker)] for price_type, ticker in tech_stock.columns},
    index=tech_stock.index,
)

## 4. Exploración de datos

En este dataset podemos ver que por cada stock, medidos una vez al día:
- **High**: el precio máximo del stock durante el día
- **Low**: el precio mínimo del stock en el día
- **Open**: el precio al comienzo del día
- **Close**: el precio al final del día
- **Volume**: el volumen de transacciones del día

In [ ]:
tech_stock.head(10)

### Selección del target

Tomamos uno de los stocks (Apple) como variable objetivo que vamos a predecir.

In [ ]:
target_col = 'Close_AAPL'
target = tech_stock[target_col]

### Visualización de la serie de tiempo

Graficamos la variación de los precios de cierre de Apple a lo largo del tiempo seleccionado.

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

sns.displot(target)
plt.show()

sns.lineplot(target)

## 5. Preparación de datos

### División del dataset

Dividimos los datos en:
- **60%** para entrenamiento
- **20%** para validación
- **20%** para test

Es importante que en series de tiempo la división sea **secuencial** (no aleatoria),
ya que queremos predecir el futuro a partir del pasado.

In [ ]:
num_train_samples = int(0.6 * len(tech_stock))  # 60% de los datos
num_val_samples = int(0.2 * len(tech_stock))  # 20% de los datos
num_test_samples = len(tech_stock) - num_train_samples - num_val_samples  # los datos restantes
print("num_train_samples:", num_train_samples)
print("num_val_samples:", num_val_samples)
print("num_test_samples:", num_test_samples)

### Estandarización

Estandarizamos los datos restando la media y dividiendo por la desviación estándar
del conjunto de entrenamiento.

In [ ]:
# Estandarizacion de los datos
mean = tech_stock[:num_train_samples].mean(axis=0)
tech_stock -= mean
std = tech_stock[:num_train_samples].std(axis=0)
tech_stock /= std

In [ ]:
# Dividir los datos
train_data = tech_stock[:num_train_samples]
valid_data = tech_stock[num_train_samples:num_train_samples + num_val_samples]
test_data = tech_stock[num_train_samples + num_val_samples:]

### Creación de datasets con ventanas temporales

Usaremos la función `timeseries_dataset_from_array` para crear los datasets.

La idea general es que al proporcionar una matriz de datos de series temporales,
`timeseries_dataset_from_array` genera **ventanas deslizantes** extraídas de la serie temporal original.

Por ejemplo, si nuestros datos son `[0 1 2 3 4 5 6]` y `sequence_length=3`,
se generarán las siguientes muestras: `[0 1 2]`, `[1 2 3]`, `[2 3 4]`, `[3 4 5]`, `[4 5 6]`.

- **sequence_length**: Longitud de las secuencias de entrada (número de pasos de tiempo)
- **batch_size**: Número de secuencias por batch

Documentación: https://www.tensorflow.org/api_docs/python/tf/keras/utils/timeseries_dataset_from_array

In [ ]:
import tensorflow as tf

batch_size = 8
sequence_length = 60


def make_dataset(data, shuffle=False):
    return tf.keras.utils.timeseries_dataset_from_array(
        data=data.drop(columns=[target_col]).values[:-1],
        targets=data[target_col].values[sequence_length:],
        sequence_length=sequence_length,
        batch_size=batch_size,
        shuffle=shuffle
    )


train_dataset = make_dataset(train_data, shuffle=True)
valid_dataset = make_dataset(valid_data, shuffle=False)
test_dataset = make_dataset(test_data, shuffle=False)

El código anterior produce un dataset en el cual la variable independiente **X** es los valores del stock
en un periodo de **60 días** y el target es el precio de cierre de Apple en el día después de la secuencia.

In [ ]:
for i, (samples, target) in zip(range(3), train_dataset):
    print(f'sample shape: {samples.shape}, target shape: {target.shape}')
    if i == 2:
        print(samples)
        print(target)

## 6. Modelo LSTM

Construimos un modelo secuencial con:
1. Una capa **LSTM** con 64 unidades y `return_sequences=True` (para pasar la secuencia completa a la siguiente capa)
2. Una segunda capa **LSTM** con 32 unidades y `return_sequences=False`
3. Una capa **Dense** con 32 neuronas y activación ReLU
4. Una capa de **salida** con 1 neurona (predicción del precio)

Usamos **MAE** (Mean Absolute Error) como función de pérdida.

In [ ]:
number_of_features = train_data.shape[1] - 1

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, number_of_features)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

# Compile
model.compile(
    optimizer='adam',
    loss='mae',  # Mean Absolute Error
    metrics=['mae']
)

model.summary()

## 7. Entrenamiento

Entrenamos el modelo por **100 épocas**. Observaremos las curvas de pérdida
para detectar posible sobreajuste.

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=100
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training MAE')
plt.plot(history.history['val_loss'], label='Validation MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE Loss')
plt.title('Training & Validation MAE over Epochs')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Evaluación del modelo

Evaluamos el modelo en el conjunto de test y generamos predicciones
para comparar visualmente con los valores reales.

In [ ]:
test_loss, test_mae = model.evaluate(test_dataset)

print(f"Test MAE: {test_mae:.4f}")

In [ ]:
import numpy as np

# Obtener todos los inputs y targets del valid_dataset
y_true = []
y_pred = []

for x_batch, y_batch in valid_dataset:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy().flatten())
    y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

## 9. Visualización de resultados

Comparamos las predicciones del modelo LSTM contra los valores reales
del conjunto de validación.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Crear DataFrame para graficar
df_plot = pd.DataFrame({
    "Real": y_true,
    "Predicción": y_pred
})

# Plot real vs predicho
sns.set(style="whitegrid")
plt.figure(figsize=(8, 5))
sns.lineplot(data=df_plot)
sns.lineplot(x=df_plot.index, y="Real", data=df_plot)

plt.title("Predicciones vs. Valores Reales")
plt.xlabel("Índice de muestra")
plt.ylabel("Valor")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Análisis del error

En este caso podemos ver que las predicciones pueden tener un sesgo,
esto se debe a que los datos de entrenamiento se tomaron de una sección con
un promedio diferente al de validación. Este es un fenómeno común en series de
tiempo financieras, donde la distribución de los datos cambia a lo largo del tiempo.